# Prompt design case study — three wordings of the same AI-researcher prompt

The same two-track figure as
[two_track_figure.ipynb](two_track_figure.ipynb), over the prompt-design runs:
rows are prompt variants rather than ablations, and each is measured against the
original AI-researcher prompt in the strip on top. `*` marks p < 0.05 (two-sided
corpus-level paired bootstrap, n=100,000).

Two forms of the thing being judged: the **plan form**, which is what the
AI-researcher prompt was written for, and the **idea form**, the idea as written.
Same prompts, same judges, same metric — only the merges differ, so the pair says
whether a wording's effect survives the form it is asked about.

The paper figure draws both at once, as four blocks on one scale; the two
single-form figures are drawn as well, for showing one on its own.

Pairwise only: these runs have no pointwise data.

Reads the `unified_*.json` chart data `merge_ablation_runs.py` writes; writes a
PDF (vector, for the paper) and a PNG per figure to
`output/figures/prompt-design/`, named `prompt-design-both_*`,
`prompt-design-plan_*` and `prompt-design-idea_*` so they sit together in one
directory. The drawing is `common.TwoTrackFigure` — this notebook is only the
wording.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "output").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.figures.common import TwoTrackFigure

print(ROOT)

## What the figures say

Everything that decides what these figures say is in the next cell. A block is a
merge, so a variant's row is the same wording scored against a different baseline
in each one.

The three figures are one figure and two copies of it: `PLAN` is written out in
full, and `IDEA` and `BOTH` are `replace(PLAN, ...)`, so rows, judges, metric,
labels and the significance key are said once and cannot drift between them.
Anything that should differ — the merges, the name it saves under — is named in
the `replace` call and nowhere else. `figure` is the directory and `file_prefix`
the filename, so all three share the former and differ in the latter.

`BOTH` is four blocks rather than two. Nothing in `TwoTrackFigure` fixes the
number: it lays out as many blocks as `tracks` has, and the block titles are the
only place a form can be named.

In [ ]:
PLAN = TwoTrackFigure(
    root=ROOT,
    figure="prompt-design",           # output subdirectory: every form shares it
    file_prefix="prompt-design-plan",  # what this form's filenames start with
    # (block title, merge dir under output/ablation_sweeps, name token)
    # The plan form: merge_ai_researcher_{hvh,vanilla_ai}-plan.
    tracks=[("Human-only", "merge_ai_researcher_hvh-plan", "human"),
            ("Human + generated", "merge_ai_researcher_vanilla_ai-plan", "vanilla")],
    # (panel name, row label), top to bottom. Named short: the left margin is
    # sized to the longest label, and on a figure of eight cells that margin is
    # what the page pays most for. The caption says what was removed from each.
    rows=[("ai_researcher_no_review", 'P4 · "Which is novel?"'),
          ("ai_researcher_comparative", 'P5 · "Which was judged\nmore novel?"')],
    # judge columns, left to right
    models=["claude-opus-4-6", "gpt-5.4"],
    metrics={"pairwise": "pairwise_accuracy"},
    # No row axis title: two rows that are both prompt variants say what the
    # axis is, and a title for them costs the margin more than it tells.
    row_axis_label="",
    # The strip is not an ablated baseline here: it is the prompt as published.
    baseline_label='P3 · "Which was judged novel?"',
    # The key to the marked cells, set in the bottom-left corner under the
    # row names. Only the delta figure marks anything; "" drops the key.
    sig_note="* p < 0.05 (two-sided paired bootstrap)",
)

# The same prompts over the idea as written rather than the plan it converts to.
# The merges are the ones configs/merge/merge_test_hvh.yaml and
# merge_test_vanilla_ai.yaml write; everything else is PLAN's, so the two forms
# can only differ in what they were run on — and in the name they are saved
# under, since they share a directory.
IDEA = replace(
    PLAN,
    file_prefix="prompt-design-idea",
    tracks=[("Human-only", "merge_test_ai_researcher_hvh", "human"),
            ("Human + generated", "merge_ai_researcher_vanilla_ai", "vanilla")],
)

# Both forms as one figure: four blocks rather than two, which pays for the row
# names, the axis title and the key once instead of twice, and puts every number
# on one scale — so plan and idea are comparable by shade and not only by
# number. The block titles name the form, since nothing else in the figure can.
#
# A title is centred on its block and may be wider than it, which only the last
# block has no room for: at 7.10in the longest one ends 0.006in past the edge
# and loses its last letter. 7.25in makes every block wide enough to hold its
# own title, and the figure still comes out narrower than one of the pair did
# before the row names were shortened.
BOTH = replace(
    PLAN,
    file_prefix="prompt-design-both",
    width=7.25,
    tracks=[("Plan · Human-only", "merge_ai_researcher_hvh-plan", "human"),
            ("Plan · Human + generated", "merge_ai_researcher_vanilla_ai-plan", "vanilla"),
            ("Idea · Human-only", "merge_test_ai_researcher_hvh", "human"),
            ("Idea · Human + generated", "merge_ai_researcher_vanilla_ai", "vanilla")],
)

# Every cell below reads this, so a figure is drawn, tabled and saved by being
# named here — and dropping one is a matter of dropping its line. BOTH is the
# one for the paper; the single-form pair is kept for showing one on its own.
FIGURES = {"Both forms": BOTH, "Plan form": PLAN, "Idea form": IDEA}

## The numbers

`fig.table(setup)` is the whole data path: one tidy row per (track, ablation,
judge), everything already in points. Print it, sort it, `.query()` it — if a
cell in a figure looks wrong, the reason is visible here, before any drawing
happens. One table per form, so the two are read against each other.

A missing panel, or a cell `common.UNSUPPORTED_CELLS` blanks (a judge that
cannot honour the ablation), gives `NaN` and draws as `—`.

In [ ]:
for form, fig in FIGURES.items():
    print(f"—— {form} ——")
    print(fig.table(next(iter(fig.metrics))).to_string(), "\n")

## The figures: what each change did

One `pcolormesh` per block, then the cell text on top of it. Diverging red→green
scale centred on zero and shared across the blocks of a figure, so a colour means
the same thing left and right; every cell also prints its number, so colour is
never the only encoding. The baseline strip is drawn outside the grid, in a shade
of its own, so it never reads as a delta of zero.

The four-block figure puts both forms on one scale, so a shade there compares
plan with idea. In the single-form pair each figure scales to its own extreme
cell, so shades are comparable within a figure but not between the two — read the
numbers when comparing those.

In [ ]:
for form, fig in FIGURES.items():
    for setup in fig.metrics:
        png = fig.draw(setup)
        print(f"{form}: {png.relative_to(ROOT)} (+ .pdf)")
        display(Image(filename=str(png)))